# 🔍 实验一：算子剖析 —— MobileNetV3 中的 Conv 与 BN

## 学习目标

1. 在 MobileNetV3 中定位所有"Conv2d 后紧跟 BatchNorm2d"的算子对
2. 理解为什么 BN 可以折叠进 Conv（算子融合）
3. 为后续"优化前 / 优化后"对比准备模型

MobileNetV3 的每个 MobileBlock 基本都包含：

- 1×1 Conv + BN + 激活（扩展层）
- Depthwise Conv + BN（深度卷积层）
- 1×1 Conv + BN + 激活（投影层）

因此整个模型中有大量 Conv+BN 算子对，非常适合做算子融合实验。

> 本 Notebook 已内嵌完整模型定义与工具代码，不需要导入任何外部 .py 文件，可直接独立运行。

In [ ]:
# ====== MobileNetV3 完整模型定义（内嵌，无需外部文件）======
import torch
import torch.nn as nn
import torch.nn.functional as F

# ====== MobileNetV3 完整模型定义（自包含，不依赖外部文件）======

def get_model_parameters(model):
    total_parameters = 0
    for layer in list(model.parameters()):
        layer_parameter = 1
        for l in list(layer.size()):
            layer_parameter *= l
        total_parameters += layer_parameter
    return total_parameters

def _make_divisible(v, divisor=8, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

def _weights_init(m):
    if isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        m.weight.data.fill_(1)
        m.bias.data.zero_()
    elif isinstance(m, nn.Linear):
        n = m.weight.size(1)
        m.weight.data.normal_(0, 0.01)
        m.bias.data.zero_()

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        return F.relu6(x + 3., inplace=self.inplace) / 6.

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        out = F.relu6(x + 3., self.inplace) / 6.
        return out * x

class SqueezeBlock(nn.Module):
    def __init__(self, exp_size, divide=4):
        super().__init__()
        self.dense = nn.Sequential(
            nn.Linear(exp_size, exp_size // divide),
            nn.ReLU(inplace=True),
            nn.Linear(exp_size // divide, exp_size),
            h_sigmoid()
        )
    def forward(self, x):
        batch, channels, height, width = x.size()
        out = F.avg_pool2d(x, kernel_size=[height, width]).view(batch, -1)
        out = self.dense(out).view(batch, channels, 1, 1)
        return out * x

class MobileBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernal_size, stride, nonLinear, SE, exp_size):
        super().__init__()
        padding = (kernal_size - 1) // 2
        self.use_connect = stride == 1 and in_channels == out_channels
        self.SE = SE
        activation = nn.ReLU if nonLinear == "RE" else h_swish
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, exp_size, 1, 1, 0, bias=False),
            nn.BatchNorm2d(exp_size), activation(inplace=True))
        self.depth_conv = nn.Sequential(
            nn.Conv2d(exp_size, exp_size, kernal_size, stride, padding, groups=exp_size),
            nn.BatchNorm2d(exp_size))
        if self.SE:
            self.squeeze_block = SqueezeBlock(exp_size)
        self.point_conv = nn.Sequential(
            nn.Conv2d(exp_size, out_channels, 1, 1, 0),
            nn.BatchNorm2d(out_channels), activation(inplace=True))
    def forward(self, x):
        out = self.depth_conv(self.conv(x))
        if self.SE:
            out = self.squeeze_block(out)
        out = self.point_conv(out)
        return x + out if self.use_connect else out

class MobileNetV3(nn.Module):
    def __init__(self, model_mode="LARGE", num_classes=1000, multiplier=1.0, dropout_rate=0.0):
        super().__init__()
        self.num_classes = num_classes
        if model_mode == "LARGE":
            layers = [
                [16, 16, 3, 1, "RE", False, 16], [16, 24, 3, 2, "RE", False, 64],
                [24, 24, 3, 1, "RE", False, 72], [24, 40, 5, 2, "RE", True, 72],
                [40, 40, 5, 1, "RE", True, 120], [40, 40, 5, 1, "RE", True, 120],
                [40, 80, 3, 2, "HS", False, 240], [80, 80, 3, 1, "HS", False, 200],
                [80, 80, 3, 1, "HS", False, 184], [80, 80, 3, 1, "HS", False, 184],
                [80, 112, 3, 1, "HS", True, 480], [112, 112, 3, 1, "HS", True, 672],
                [112, 160, 5, 1, "HS", True, 672], [160, 160, 5, 2, "HS", True, 672],
                [160, 160, 5, 1, "HS", True, 960],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(160 * multiplier), _make_divisible(960 * multiplier), 1, 1),
                nn.BatchNorm2d(_make_divisible(960 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(960 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        elif model_mode == "SMALL":
            layers = [
                [16, 16, 3, 2, "RE", True, 16], [16, 24, 3, 2, "RE", False, 72],
                [24, 24, 3, 1, "RE", False, 88], [24, 40, 5, 2, "RE", True, 96],
                [40, 40, 5, 1, "RE", True, 240], [40, 40, 5, 1, "RE", True, 240],
                [40, 48, 5, 1, "HS", True, 120], [48, 48, 5, 1, "HS", True, 144],
                [48, 96, 5, 2, "HS", True, 288], [96, 96, 5, 1, "HS", True, 576],
                [96, 96, 5, 1, "HS", True, 576],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(96 * multiplier), _make_divisible(576 * multiplier), 1, 1),
                SqueezeBlock(_make_divisible(576 * multiplier)),
                nn.BatchNorm2d(_make_divisible(576 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(576 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        self.apply(_weights_init)
    def forward(self, x):
        out = self.block(self.init_conv(x))
        out = self.out_conv1(out)
        b, c, h, w = out.size()
        return self.out_conv2(F.avg_pool2d(out, [h, w])).view(b, -1)

# ====== 实例化模型并统计算子 ======
import torch
import torch.nn as nn

model = MobileNetV3(model_mode="LARGE", num_classes=200)
print(f"参数量: {get_model_parameters(model):,}")

conv_count = sum(1 for m in model.modules() if isinstance(m, nn.Conv2d))
bn_count = sum(1 for m in model.modules() if isinstance(m, nn.BatchNorm2d))
print(f"Conv2d 数量: {conv_count}")
print(f"BatchNorm2d 数量: {bn_count}")

In [ ]:
def find_conv_bn_pairs(module, prefix=""):
    pairs = []
    for name, child in module.named_children():
        path = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Sequential):
            children = list(child)
            for i in range(len(children) - 1):
                if isinstance(children[i], nn.Conv2d) and isinstance(children[i + 1], nn.BatchNorm2d):
                    pairs.append((f"{path}[{i}]", children[i], children[i + 1]))
        pairs.extend(find_conv_bn_pairs(child, path))
    return pairs

pairs = find_conv_bn_pairs(model)
print(f"可融合的 Conv+BN 算子对: {len(pairs)}")
print()
print(f"{'序号':<4}{'位置':<36}{'输入通道':<10}{'输出通道':<10}{'kernel':<8}{'groups':<8}")
for i, (path, conv, bn) in enumerate(pairs[:12], 1):
    print(f"{i:<4}{path:<36}{conv.in_channels:<10}{conv.out_channels:<10}{conv.kernel_size[0]:<8}{conv.groups:<8}")
if len(pairs) > 12:
    print(f"... 共 {len(pairs)} 对")

## 为什么 Conv+BN 可以融合

推理阶段 BatchNorm 使用固定的 running_mean 与 running_var：

$$y = \gamma \cdot \frac{x - \text{running\_mean}}{\sqrt{\text{running\_var} + \epsilon}} + \beta$$

设卷积输出为 $z = W \cdot x + b$，则：

$$y = \gamma \cdot \frac{W \cdot x + b - \text{running\_mean}}{\sqrt{\text{running\_var} + \epsilon}} + \beta$$

$$y = \left(\frac{\gamma \cdot W}{\sqrt{\text{running\_var} + \epsilon}}\right) \cdot x + \frac{\gamma \cdot (b - \text{running\_mean})}{\sqrt{\text{running\_var} + \epsilon}} + \beta$$

所以可以构造新的卷积参数：

$$W' = \frac{\gamma \cdot W}{\sqrt{\text{running\_var} + \epsilon}}, \quad b' = \frac{\gamma \cdot (b - \text{running\_mean})}{\sqrt{\text{running\_var} + \epsilon}} + \beta$$

融合后推理只需要一次 Conv，BatchNorm 算子被完全消除。

## 重要前提

- 该融合只适用于 **eval / 推理模式**，因为训练时 BN 使用当前 batch 的均值方差，无法提前折进权重
- 融合是**数学等价变换**，不会改变模型输出（浮点舍入误差除外）
- 在昇腾侧，CANN / ATC 在构图与编译阶段也会做类似的算子融合；本实验在 PyTorch 模型层复现同一思路，方便直接量化收益

## 本实验的算子选择

本实验只围绕 **Conv2d + BatchNorm2d 这一个算子对** 展开优化，不涉及其他注意力或扩展算子。
选择它的原因：

- 数量多：MobileNetV3-Large 中有 40+ 个可融合的 Conv+BN 算子对，优化收益可被量化
- 数学等价：BN 折叠不改变模型输出，便于做正确性校验
- 通用性强：Conv+BN 是 CNN 中最常见的算子组合，在昇腾 CANN / ATC 图编译中也会被自动融合

## 课后练习

1. (单选题) 如果 running_var 被错误设为 0 且 eps 很小，BN 折叠的主要风险是？
   - A. 权重被放大导致数值不稳定
   - B. 模型必然无法加载
   - C. 输出 shape 改变
   - D. 无风险

2. (单选题) 融合 Depthwise Conv 时忘记保持 groups 会造成？
   - A. 输出错误或 shape 不匹配
   - B. 精度更高
   - C. 无影响
   - D. 只影响速度

3. (单选题) find_conv_bn_pairs 允许 Conv+BN 后还有 ReLU 的原因是？
   - A. 融合不改变激活层，应保留后续结构
   - B. ReLU 会被 BN 吸收
   - C. 无需保留
   - D. 只统计前两层

4. (多选题) eval 折叠使用 BN 的哪些参数？
   - A. weight
   - B. bias
   - C. running_mean
   - D. running_var

5. (多选题) 以下哪些错误会导致融合前后输出不一致？
   - A. 未调用 model.eval()
   - B. 误用当前 batch 统计量
   - C. 未复制原始 conv 权重
   - D. 缺少 torch.no_grad()

6. (判断题) BN 折叠是数学等价变换，融合前后输出逐 bit 完全一致。

7. (判断题) running_mean 与 running_var 在 eval 模式下不会更新。

8. (填空题) 折叠后新权重 W' = ____。

9. (填空题) 折叠后新偏置 b' = ____。

10. (简答题) 为什么训练模式不能直接折叠 BN？

11. (简答题) 为什么递归融合必须 setattr 写回？

12. (代码设计题) 实现 find_conv_bn_pairs(model)，返回所有 Conv2d 后紧跟 BatchNorm2d 的 (父模块, conv, bn)。

13. (单选题) nn.BatchNorm2d 的 weight/bias 参数 shape 是？
   - A. (C,)
   - B. (1,C,1,1)
   - C. (C,1,1)
   - D. (N,C)

14. (多选题) 融合后应保留的模块包括？
   - A. 激活层
   - B. BatchNorm
   - C. SE 模块
   - D. 池化层

15. (简答题) 融合前 total=96、融合后 total=49，请解释 Conv 数量不变而 BN 全部消失。

> 参考答案见 answer/06.03_operator_analysis_answer.ipynb。